# TCN Inter-Patient Experiment: RevIN vs No-RevIN (H=10)

**Objective**: Test whether Reversible Instance Normalization (RevIN) improves TCN
forecasting on **unseen patients** (inter-patient generalisation).

**Setup**:
- **Architecture**: Same TCN as `tcn-nu2_MINE.ipynb` — 3 residual blocks, 64 filters, kernel 3, batch 256
- **Horizon**: H=10 only
- **Split**: 15 train patients → 6 unseen test patients (inter-patient)
- **Normalization**: No MinMax — RevIN normalizes per-window at runtime

**Models compared**:
1. **TCN (no RevIN)** — raw signal, no normalization
2. **TCN + RevIN** — per-instance normalization + denormalization

Metrics: RMSE · MAE · R² · Training Time

In [1]:
# ── Install & Imports ────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'wfdb', 'tqdm', 'seaborn', '--quiet'])

import os, time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)

SEED = 42
np.random.seed(SEED)
print('All imports OK.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 10.6 MB/s eta 0:00:00
All imports OK.


In [2]:
# ── TensorFlow Setup ─────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, Add, Activation, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s): {[g.name for g in gpus]}')
else:
    print('No GPU — training will be slow.')

GPU(s): ['/physical_device:GPU:0', '/physical_device:GPU:1']


## 1. Configuration

In [3]:
# ── Dataset path ─────────────────────────────────────────────────────────────
DATA_DIR = r"/kaggle/input/datasets/rracer17/mit-bih-mitdb/mit-bih-arrhythmia-database-1.0.0"

# ── Paper constants ──────────────────────────────────────────────────────────
FS           = 360
TOTAL_STEPS  = 100_000
LOOKBACK     = 10

# ── All 21 patients (excluding 111 & 118) ────────────────────────────────────
PATIENTS = [
    '100', '101', '102', '103', '104', '105',
    '106', '107', '108', '109', '112', '113',
    '114', '115', '116', '117', '119',
    '121', '122', '123', '124'
]
assert len(PATIENTS) == 21

# ── Inter-patient split (same as LSTM experiment) ────────────────────────────
TRAIN_PATIENTS = [
    '100', '101', '102', '103', '104', '105', '106',
    '107', '108', '109', '112', '113', '114', '115', '116'
]
TEST_PATIENTS = ['117', '119', '121', '122', '123', '124']
assert len(TRAIN_PATIENTS) + len(TEST_PATIENTS) == len(PATIENTS)

# ── Experiment config ────────────────────────────────────────────────────────
HORIZON       = 10            # single horizon for this experiment
VAL_RATIO     = 0.2           # 80/20 split within each train patient
EPOCHS        = 30
PATIENCE      = 5
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3

# ── TCN architecture (identical to tcn-nu2_MINE.ipynb "Our Config") ──────────
NUM_FILTERS   = 64
KERNEL_SIZE   = 3
NUM_BLOCKS    = 3             # dilations 1, 2, 4
DROPOUT_RATE  = 0.1

print(f'Train patients ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}')
print(f'Test  patients ({len(TEST_PATIENTS)}):  {TEST_PATIENTS}')
print(f'Horizon:  {HORIZON}')
print(f'Lookback: {LOOKBACK}')
print(f'TCN: {NUM_BLOCKS} blocks, {NUM_FILTERS} filters, kernel {KERNEL_SIZE}')
print(f'Training: {EPOCHS} epochs, patience {PATIENCE}, batch {BATCH_SIZE}')

Train patients (15): ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '112', '113', '114', '115', '116']
Test  patients (6):  ['117', '119', '121', '122', '123', '124']
Horizon:  10
Lookback: 10
TCN: 3 blocks, 64 filters, kernel 3
Training: 30 epochs, patience 5, batch 256


## 2. Data Loading (Raw — No Normalization)

In [4]:
def load_ecg_signal(record_id, data_dir, n_steps=100_000):
    """Load MLII lead from MIT-BIH, truncate/pad to n_steps."""
    path = os.path.join(data_dir, record_id)
    rec  = wfdb.rdrecord(path)
    sig_names_upper = [s.upper() for s in rec.sig_name]
    ch = sig_names_upper.index('MLII') if 'MLII' in sig_names_upper else 0
    signal = rec.p_signal[:, ch].astype(np.float32)
    if len(signal) < n_steps:
        pad = np.full(n_steps - len(signal), signal[-1], dtype=np.float32)
        signal = np.concatenate([signal, pad])
    return signal[:n_steps]


def make_sequences(signal, lookback, horizon):
    """X = lookback window, y = next H steps (Multi-Output)."""
    X, y = [], []
    for i in range(len(signal) - lookback - horizon + 1):
        X.append(signal[i : i + lookback])
        y.append(signal[i + lookback : i + lookback + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


print('Preprocessing functions defined.')

Preprocessing functions defined.


In [5]:
# ── Load raw signals (NO normalization — RevIN handles it) ────────────────────
train_signals_raw = {}
test_signals_raw  = {}

print('Loading train patient signals (raw)...')
for rid in tqdm(TRAIN_PATIENTS, desc='Train'):
    train_signals_raw[rid] = load_ecg_signal(rid, DATA_DIR, TOTAL_STEPS)

print('Loading test patient signals (raw)...')
for rid in tqdm(TEST_PATIENTS, desc='Test'):
    test_signals_raw[rid] = load_ecg_signal(rid, DATA_DIR, TOTAL_STEPS)

# ── Show distribution shift between patients ─────────────────────────────────
print(f"\n--- Signal Statistics (inter-patient distribution shift) ---")
print(f"{'Patient':>8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10} {'Split':>8}")
print("-" * 58)
for rid in TRAIN_PATIENTS:
    s = train_signals_raw[rid]
    print(f"{rid:>8} {s.mean():>10.3f} {s.std():>10.3f} "
          f"{s.min():>10.3f} {s.max():>10.3f} {'TRAIN':>8}")
for rid in TEST_PATIENTS:
    s = test_signals_raw[rid]
    print(f"{rid:>8} {s.mean():>10.3f} {s.std():>10.3f} "
          f"{s.min():>10.3f} {s.max():>10.3f} {'TEST':>8}")

Loading train patient signals (raw)...


Train:   0%|          | 0/15 [00:00<?, ?it/s]

Loading test patient signals (raw)...


Test:   0%|          | 0/6 [00:00<?, ?it/s]


--- Signal Statistics (inter-patient distribution shift) ---
 Patient       Mean        Std        Min        Max    Split
----------------------------------------------------------
     100     -0.322      0.176     -0.695      1.245    TRAIN
     101     -0.286      0.335     -3.175      2.420    TRAIN
     102     -0.257      0.189     -1.410      1.375    TRAIN
     103     -0.226      0.322     -0.770      2.120    TRAIN
     104     -0.231      0.296     -1.935      1.975    TRAIN
     105     -0.216      0.314     -0.795      1.970    TRAIN
     106     -0.170      0.350     -1.520      2.200    TRAIN
     107     -0.239      0.814     -3.105      2.800    TRAIN
     108     -0.229      0.185     -1.390      1.080    TRAIN
     109     -0.229      0.434     -1.605      1.770    TRAIN
     112     -0.854      0.223     -1.685      0.575    TRAIN
     113     -0.147      0.450     -1.225      2.550    TRAIN
     114      0.049      0.305     -2.310      2.450    TRAIN
     115   

In [6]:
# ── Build pooled training data ────────────────────────────────────────────────

def build_inter_datasets(signals_dict, lookback, horizon, val_ratio=0.2):
    """Pool sequences from multiple patients. No normalization applied."""
    X_tr_all, y_tr_all = [], []
    X_vl_all, y_vl_all = [], []
    for rid, signal in signals_dict.items():
        split_idx = int(len(signal) * (1 - val_ratio))
        X_tr, y_tr = make_sequences(signal[:split_idx], lookback, horizon)
        X_vl, y_vl = make_sequences(signal[split_idx:], lookback, horizon)
        X_tr_all.append(X_tr); y_tr_all.append(y_tr)
        X_vl_all.append(X_vl); y_vl_all.append(y_vl)
    return (np.concatenate(X_tr_all), np.concatenate(y_tr_all),
            np.concatenate(X_vl_all), np.concatenate(y_vl_all))


X_train, y_train, X_val, y_val = build_inter_datasets(
    train_signals_raw, LOOKBACK, HORIZON, VAL_RATIO
)

# Per-patient test sets (each patient evaluated separately)
test_datasets = {}
for rid, signal in test_signals_raw.items():
    X_te, y_te = make_sequences(signal, LOOKBACK, HORIZON)
    test_datasets[rid] = (X_te, y_te)

print(f'Train samples: {len(X_train):,}')
print(f'Val   samples: {len(X_val):,}')
for rid in TEST_PATIENTS:
    print(f'Test {rid}: {len(test_datasets[rid][0]):,} samples')

Train samples: 1,199,715
Val   samples: 299,715
Test 117: 99,981 samples
Test 119: 99,981 samples
Test 121: 99,981 samples
Test 122: 99,981 samples
Test 123: 99,981 samples
Test 124: 99,981 samples


## 3. TCN Architecture

In [7]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    return Add()([x, out])


def build_tcn(lookback, output_size):
    """Plain TCN (no RevIN) — identical to tcn-nu2_MINE.ipynb."""
    inp = Input(shape=(lookback, 1))
    x = inp
    for i in range(NUM_BLOCKS):
        x = residual_block(x, NUM_FILTERS, KERNEL_SIZE, 2 ** i, DROPOUT_RATE)
    x = x[:, -1, :]  # last time-step
    x = Dense(NUM_FILTERS, activation='relu')(x)
    out = Dense(output_size)(x)
    model = Model(inp, out, name=f'TCN_out{output_size}')
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss='mse')
    return model


print('TCN architecture defined (identical to tcn-nu2_MINE.ipynb).')

TCN architecture defined (identical to tcn-nu2_MINE.ipynb).


## 4. RevIN (Reversible Instance Normalization) — Keras

Adapted from the PyTorch RevIN used in the LSTM inter-patient experiment.

**Forward pass:**
1. Compute per-window mean & std from input `(batch, lookback, 1)`
2. Normalize: `x_norm = (x - mean) / std * γ + β`
3. TCN backbone: `out = TCN(x_norm)` → `(batch, horizon)`
4. Denormalize: `out = (out - β) / γ * std + mean`

Learnable affine parameters `γ` (gamma) and `β` (beta) are included, matching the PyTorch implementation.

In [8]:
class TCNWithRevIN(tf.keras.Model):
    """TCN wrapped with Reversible Instance Normalization.

    Normalizes each input window by its own mean/std before the TCN,
    then denormalizes the output back to the original scale.
    """

    def __init__(self, lookback, output_size, eps=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps
        # Learnable affine parameters (matches PyTorch RevIN)
        self.revin_gamma = self.add_weight(
            
            name='revin_gamma', shape=(1, 1, 1), initializer='ones', trainable=True)
        self.revin_beta = self.add_weight(
            name='revin_beta', shape=(1, 1, 1), initializer='zeros', trainable=True)
        # TCN backbone — identical architecture to build_tcn()
        self.backbone = self._build_backbone(lookback, output_size)

    def _build_backbone(self, lookback, output_size):
        inp = Input(shape=(lookback, 1))
        x = inp
        for i in range(NUM_BLOCKS):
            x = residual_block(x, NUM_FILTERS, KERNEL_SIZE, 2 ** i, DROPOUT_RATE)
        x = x[:, -1, :]
        x = Dense(NUM_FILTERS, activation='relu')(x)
        out = Dense(output_size)(x)
        return Model(inp, out, name='tcn_backbone')

    def call(self, x, training=False):
        # x: (batch, lookback, 1)
        # ── RevIN: Normalize ──
        mean = tf.reduce_mean(x, axis=1, keepdims=True)           # (batch, 1, 1)
        std  = tf.math.reduce_std(x, axis=1, keepdims=True) + self.eps
        x_norm = (x - mean) / std
        x_norm = x_norm * self.revin_gamma + self.revin_beta

        # ── TCN backbone ──
        out = self.backbone(x_norm, training=training)             # (batch, horizon)

        # ── RevIN: Denormalize ──
        out = tf.expand_dims(out, -1)                              # (batch, horizon, 1)
        out = (out - self.revin_beta) / self.revin_gamma
        out = out * std + mean                                     # broadcast (batch,1,1)
        out = tf.squeeze(out, -1)                                  # (batch, horizon)
        return out


# Sanity check
_test_model = TCNWithRevIN(LOOKBACK, HORIZON)
_test_input = tf.random.normal((2, LOOKBACK, 1))
_test_out   = _test_model(_test_input)
print(f'TCN+RevIN: input {_test_input.shape} -> output {_test_out.shape}')

# Parameter comparison
_plain = build_tcn(LOOKBACK, HORIZON)
plain_params = _plain.count_params()
revin_params = sum(np.prod(v.shape) for v in _test_model.trainable_variables)
print(f'TCN (no RevIN): {plain_params:,} params')
print(f'TCN + RevIN:    {revin_params:,} params  (+2 affine)')
del _test_model, _test_input, _test_out, _plain
tf.keras.backend.clear_session()

I0000 00:00:1786730742.271686      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786730742.274503      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


TCN+RevIN: input (2, 10, 1) -> output (2, 10)
TCN (no RevIN): 68,490 params
TCN + RevIN:    67,724 params  (+2 affine)


## 5. Metrics

In [9]:
def compute_metrics(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    return {
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'MAE':  float(mean_absolute_error(yt, yp)),
        'R2':   float(r2_score(yt, yp)),
    }

print('Metrics function defined.')

Metrics function defined.


## 6. Training: TCN (no RevIN) vs TCN + RevIN

Both models trained on the same pooled raw data from 15 train patients.
Evaluated on each of the 6 unseen test patients separately.

In [10]:
def train_model(model, X_tr, y_tr, X_vl, y_vl):
    """Train with early stopping + LR reduction."""
    X_tr_3d = X_tr.reshape(-1, X_tr.shape[1], 1)
    X_vl_3d = X_vl.reshape(-1, X_vl.shape[1], 1)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=max(PATIENCE // 2, 2), min_lr=1e-6),
    ]
    history = model.fit(
        X_tr_3d, y_tr, validation_data=(X_vl_3d, y_vl),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=2
    )
    return history


def evaluate_on_patients(model, test_datasets):
    """Evaluate on each test patient; return per-patient metrics."""
    patient_metrics = []
    for rid, (X_te, y_te) in test_datasets.items():
        X_te_3d = X_te.reshape(-1, X_te.shape[1], 1)
        preds = model.predict(X_te_3d, batch_size=2048, verbose=0)
        m = compute_metrics(y_te, preds)
        m['Patient'] = rid
        patient_metrics.append(m)
    return patient_metrics


# ── Run experiment ───────────────────────────────────────────────────────────

inter_results = {}

for model_name in ['TCN (no RevIN)', 'TCN + RevIN']:
    print(f'\n{"=" * 70}')
    print(f'  {model_name} — H={HORIZON}')
    print(f'{"=" * 70}')

    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    if model_name == 'TCN + RevIN':
        model = TCNWithRevIN(LOOKBACK, HORIZON)
        model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss='mse')
    else:
        model = build_tcn(LOOKBACK, HORIZON)

    t0 = time.time()
    history = train_model(model, X_train, y_train, X_val, y_val)
    train_time = time.time() - t0

    patient_metrics = evaluate_on_patients(model, test_datasets)

    print(f'\n  Per-patient results:')
    for pm in patient_metrics:
        print(f'    {pm["Patient"]}: RMSE={pm["RMSE"]:.4f}  '
              f'MAE={pm["MAE"]:.4f}  R\u00b2={pm["R2"]:.4f}')

    avg_rmse = np.mean([m['RMSE'] for m in patient_metrics])
    avg_mae  = np.mean([m['MAE']  for m in patient_metrics])
    avg_r2   = np.mean([m['R2']   for m in patient_metrics])

    inter_results[model_name] = {
        'avg_RMSE': round(avg_rmse, 4),
        'avg_MAE':  round(avg_mae, 4),
        'avg_R2':   round(avg_r2, 4),
        'Time_s':   round(train_time, 1),
        'per_patient': patient_metrics,
    }
    print(f'\n  AVG >> RMSE={avg_rmse:.4f}  MAE={avg_mae:.4f}  R\u00b2={avg_r2:.4f}  '
          f'({train_time:.1f}s)')

    del model
    tf.keras.backend.clear_session()
    gc.collect()

print('\n\nInter-patient experiment complete!')


  TCN (no RevIN) — H=10
Epoch 1/30


I0000 00:00:1786730757.120190      76 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4687/4687 - 40s - 8ms/step - loss: 0.0393 - val_loss: 0.0260 - learning_rate: 0.0010
Epoch 2/30
4687/4687 - 21s - 5ms/step - loss: 0.0250 - val_loss: 0.0248 - learning_rate: 0.0010
Epoch 3/30
4687/4687 - 21s - 5ms/step - loss: 0.0231 - val_loss: 0.0229 - learning_rate: 0.0010
Epoch 4/30
4687/4687 - 21s - 5ms/step - loss: 0.0221 - val_loss: 0.0222 - learning_rate: 0.0010
Epoch 5/30
4687/4687 - 21s - 5ms/step - loss: 0.0216 - val_loss: 0.0219 - learning_rate: 0.0010
Epoch 6/30
4687/4687 - 21s - 5ms/step - loss: 0.0212 - val_loss: 0.0222 - learning_rate: 0.0010
Epoch 7/30
4687/4687 - 21s - 5ms/step - loss: 0.0209 - val_loss: 0.0215 - learning_rate: 0.0010
Epoch 8/30
4687/4687 - 21s - 5ms/step - loss: 0.0206 - val_loss: 0.0210 - learning_rate: 0.0010
Epoch 9/30
4687/4687 - 21s - 5ms/step - loss: 0.0204 - val_loss: 0.0211 - learning_rate: 0.0010
Epoch 10/30
4687/4687 - 21s - 5ms/step - loss: 0.0201 - val_loss: 0.0211 - learning_rate: 0.0010
Epoch 11/30
4687/4687 - 21s - 5ms/step - loss: 0.0

## 7. Results: RevIN vs No-RevIN Comparison (H=10)

In [11]:
# ── Summary Table ────────────────────────────────────────────────────────────

print("=" * 90)
print("  INTER-PATIENT RESULTS: TCN — RevIN vs No-RevIN  (H=10)")
print("=" * 90)

header = f"{'Model':>18} | {'RMSE':>8} | {'MAE':>8} | {'R\u00b2':>8} | {'Time (s)':>9}"
print(header)
print("-" * len(header))

for model_name in ['TCN (no RevIN)', 'TCN + RevIN']:
    r = inter_results[model_name]
    print(f"{model_name:>18} | {r['avg_RMSE']:>8.4f} | "
          f"{r['avg_MAE']:>8.4f} | {r['avg_R2']:>8.4f} | "
          f"{r['Time_s']:>9.1f}")

# ── Improvement ──────────────────────────────────────────────────────────────
nr = inter_results['TCN (no RevIN)']
wr = inter_results['TCN + RevIN']
rmse_pct = (nr['avg_RMSE'] - wr['avg_RMSE']) / nr['avg_RMSE'] * 100
mae_pct  = (nr['avg_MAE']  - wr['avg_MAE'])  / nr['avg_MAE']  * 100
r2_diff  = wr['avg_R2'] - nr['avg_R2']

print(f"\n{'\u2500' * 50}")
print(f"RevIN improvement over no-RevIN:")
print(f"  RMSE: {rmse_pct:>+.1f}%")
print(f"  MAE:  {mae_pct:>+.1f}%")
print(f"  R\u00b2:   {r2_diff:>+.4f}")

# ── Per-Patient Breakdown ────────────────────────────────────────────────────
print(f"\n\n{'=' * 90}")
print("  PER-PATIENT BREAKDOWN (H=10, unseen test patients)")
print(f"{'=' * 90}")

comp_rows = []
for i, rid in enumerate(TEST_PATIENTS):
    nr_m = inter_results['TCN (no RevIN)']['per_patient'][i]
    wr_m = inter_results['TCN + RevIN']['per_patient'][i]
    comp_rows.append({
        'Patient': rid,
        'RMSE (no RevIN)': nr_m['RMSE'],
        'RMSE (RevIN)':    wr_m['RMSE'],
        'MAE (no RevIN)':  nr_m['MAE'],
        'MAE (RevIN)':     wr_m['MAE'],
        'R\u00b2 (no RevIN)':   nr_m['R2'],
        'R\u00b2 (RevIN)':      wr_m['R2'],
    })

df_comp = pd.DataFrame(comp_rows)
display(df_comp.set_index('Patient').round(4))

# ── Save to CSV ──────────────────────────────────────────────────────────────
df_comp.to_csv('inter_patient_tcn_revin_results.csv', index=False)
print(f"\nResults saved \u2192 inter_patient_tcn_revin_results.csv")

  INTER-PATIENT RESULTS: TCN — RevIN vs No-RevIN  (H=10)
             Model |     RMSE |      MAE |       R² |  Time (s)
---------------------------------------------------------------
    TCN (no RevIN) |   0.1870 |   0.0760 |   0.7318 |     650.5
       TCN + RevIN |   0.1742 |   0.0682 |   0.7585 |     642.4

──────────────────────────────────────────────────
RevIN improvement over no-RevIN:
  RMSE: +6.8%
  MAE:  +10.3%
  R²:   +0.0267


  PER-PATIENT BREAKDOWN (H=10, unseen test patients)


,RMSE (no RevIN),RMSE (RevIN),MAE (no RevIN),MAE (RevIN),R² (no RevIN),R² (RevIN)
Patient,,,,,,
117,0.1123,0.1293,0.0563,0.0576,0.7811,0.7100
119,0.2717,0.2480,0.1023,0.0904,0.7430,0.7858
121,0.1143,0.1069,0.0509,0.0475,0.8369,0.8574
122,0.2291,0.1968,0.1007,0.0824,0.6051,0.7088
123,0.1577,0.1686,0.0646,0.0632,0.7418,0.7050
124,0.2369,0.1956,0.0810,0.0680,0.6826,0.7838



Results saved → inter_patient_tcn_revin_results.csv
